<a href="https://colab.research.google.com/github/csugasi/DATASCI266-Banking77-Encoder-vs-Decoder/blob/shuchi%2Fdev/notebooks/qwen_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata
import os

token = userdata.get('GH_TOKEN')
repo  = "csugasi/DATASCI266-Banking77-Encoder-vs-Decoder"
path  = "/content/repo"

if not os.path.exists(path):
    !git clone -q -b shuchi/dev https://{token}@github.com/{repo}.git {path}

%cd {path}
!git log --oneline -3
!ls data/splits

/content/repo
ffafaaf (HEAD -> shuchi/dev, origin/shuchi/dev) Add Qwen decoder notebook
00f1409 edited .gitignore to exclude model weights and env
074307e (origin/main, origin/HEAD) Add 50% proportional subset for 3 seeds
full_train.parquet	 train_k20_seed0.parquet  train_k3_seed2.parquet
manifest.json		 train_k20_seed1.parquet  train_k5_seed0.parquet
train_k10_seed0.parquet  train_k20_seed2.parquet  train_k5_seed1.parquet
train_k10_seed1.parquet  train_k30_seed0.parquet  train_k5_seed2.parquet
train_k10_seed2.parquet  train_k30_seed1.parquet  train_p50_seed0.parquet
train_k1_seed0.parquet	 train_k30_seed2.parquet  train_p50_seed1.parquet
train_k1_seed1.parquet	 train_k3_seed0.parquet   train_p50_seed2.parquet
train_k1_seed2.parquet	 train_k3_seed1.parquet   val.parquet


In [3]:
import pandas as pd, json

val   = pd.read_parquet("data/splits/val.parquet")
test  = pd.read_parquet("data/raw/test.parquet")
k5    = pd.read_parquet("data/splits/train_k5_seed0.parquet")
labels = json.load(open("data/raw/label_names.json"))

print("val:", val.shape, "| test:", test.shape, "| k5 seed0:", k5.shape, "| labels:", len(labels))
print(k5.head(3))

val: (385, 3) | test: (3080, 2) | k5 seed0: (385, 3) | labels: 77
   id                                               text  label
0  41     Where is my credit card that was to be mailed?     11
1  59   I want to find out what happened to my new card?     11
2  72  How can I see the progress of delivery of the ...     11


In [1]:
!nvidia-smi

Sun Nov  1 02:29:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q peft bitsandbytes accelerate

import torch, transformers, peft, bitsandbytes, accelerate
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__, "| accelerate", accelerate.__version__)

torch 2.11.0+cu130 | GPU: Tesla T4
transformers 5.21.0 | peft 0.21.2 | bitsandbytes 0.50.2 | accelerate 1.15.0
